# Automated follow-up: checking every survivor before it reaches the report

Notebook 9 developed the follow-up checks by hand, one candidate at a time: does the signal
come back in the star's other sectors, does the flux disappear from the target's pixels, does a
physical transit fit give the right stellar density, and what is the false-positive probability.
For sector 104 the result was sobering. Of the 13 signals that passed every vetting test and
matched no known object, **12 were explained by these checks**: most were eclipsing binaries on
neighbouring stars, the rest did not repeat or were systematics.

That makes the follow-up part of the pipeline, not an optional extra. The vetting tests of
notebook 5 only see the target's own light curve. They cannot tell that a perfect-looking transit
comes from a star 40″ away, or that it never happens again in four other sectors. A ranked report
that puts such a signal at the top wastes the reader's time, and a pipeline run on many sectors
cannot rely on someone running notebook 9 by hand.

This notebook explains the follow-up **stage** (`followup` in `src/cli.py`, the code in
`src/followup.py`):

1. which signals it checks;
2. what it runs on each, and in which order, so the expensive steps are spent only where they can
   change the answer;
3. the verdict rules, which turn the measurements into one word per candidate;
4. how it runs unattended: in parallel, resumably, and without one failure stopping the run;
5. the result for sector 104, compared with the verdicts reached by hand in
   `docs/REVIEW_sector104.md`.

The physics of each check is in notebook 9; here it is about turning them into a pipeline step.

In [1]:
import json
import os
import sys
import warnings
from pathlib import Path

import dotenv
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from src import followup as fu

warnings.filterwarnings("ignore")
ROOT = Path("..")
dotenv.load_dotenv(ROOT / ".env")
SECTOR = int(os.environ["SECTOR"])
DATA_DIR = ROOT / "data"
FU = DATA_DIR / "followup"
fu.DATA_DIR, fu.FOLLOWUP_DIR = DATA_DIR, FU  # .env's DATA_DIR is relative to the repository root, not to notebooks/
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)
known = pd.read_parquet(DATA_DIR / "search" / f"sector_{SECTOR}_known.parquet")

## 1. Which signals are followed up

After vetting (notebook 5) and the known-object match (notebook 6), every signal that passes all
hard tests has a `known_class`. The follow-up checks only the ones nothing explains yet:

| class | followed up? | why |
|---|---|---|
| `new` | yes | the point of the pipeline |
| `new_in_known_system` | yes | a new signal around a star with a known (often non-transiting) planet |
| `known_planet`, `toi`, `ctoi` | no | already a planet or a candidate someone else follows |
| `known_eb`, `neighbour` | no | already explained: a catalogued binary, or the signal of a neighbouring star found in its own light curve |

`fu.survivors()` makes this selection. Signals failing vetting are not followed up: the thresholds
were tuned on injections (notebook 8) and the follow-up is not a second chance for them.

In [2]:
surv = fu.survivors(known)
print(f"{known.passes_all.sum()} signals pass vetting; by class:", known[known.passes_all].known_class.value_counts().to_dict())
print(f"{len(surv)} are followed up")
surv[["tic_id", "known_class", "period", "depth_ppm", "duration_h", "rp_earth", "snr", "SDE", "flags"]].round(3).reset_index(drop=True)

31 signals pass vetting; by class: {'new': 13, 'neighbour': 9, 'known_planet': 8, 'ctoi': 1}
13 are followed up


,tic_id,known_class,period,depth_ppm,duration_h,rp_earth,snr,SDE,flags
0,229804549,new,0.732,2385.447,1.532,2.755,17.300,13.530,centroid
1,218663210,new,0.788,7855.965,0.979,2.879,12.051,19.359,"v_shaped,centroid"
2,306101698,new,2.879,4882.404,2.010,3.066,10.459,17.019,centroid
3,61615837,new,1.108,3557.684,1.539,2.359,10.261,16.326,centroid
4,214311908,new,5.692,13792.993,1.138,4.385,10.162,14.305,"centroid,contamination"
5,161313862,new,0.293,2705.093,0.585,1.230,7.906,15.171,centroid
6,372868160,new,4.513,2052.967,1.626,1.483,7.565,10.738,"centroid,contamination"
7,88516648,new,1.214,847.340,1.546,1.805,7.163,12.233,centroid
8,389127976,new,0.576,30287.832,1.273,5.930,7.132,10.380,"centroid,contamination,variable_star"
9,424913827,new,1.573,676.383,1.662,0.923,7.052,12.327,centroid


## 2. What runs on each candidate, and in which order

The checks differ by orders of magnitude in cost, so the stage runs the cheap ones first and stops
as soon as the signal is explained:

| step | function | cost | question |
|---|---|---|---|
| all sectors | `fetch_sectors` | download, ~1 min | which sectors and products exist for this star? |
| repetition | `sector_search`, `alias_search`, `repetition` | TLS in a narrow window per sector, ~1–5 min | does the signal come back where it could have been seen? Is it deeper at 2P (an EB found at half its period)? |
| pixels | `difference_image` | TESScut cutout, ~1 min | does the flux disappear from the target or a neighbour? |
| neighbours | `gaia_neighbours` | Gaia query, seconds | which nearby stars are bright enough to make the dip at all? |
| *cheap rejection* | `_cheap_reject` | – | if the above already explain it (off target, not repeating, EB at 2P, or no sector can test it), stop here |
| transit fit | `fit_transit` | MCMC, ~5–15 min | radius, impact parameter, and the stellar density the shape implies |
| FPP | `fpp` | TRICERATOPS with TRILEGAL background stars, ~5–20 min | how likely is a planet on the target against all false-positive scenarios? |

In sector 104, the cheap rejection spares the fit and FPP for most candidates: an eclipsing binary
on a neighbour is already settled by its difference image.

Two rules make re-running cheap:

- **resume**: a candidate whose `summary.json` was written by the current `FOLLOWUP_VERSION`, for
  the same period and T0, with a verdict, is not redone (`fu.up_to_date`). Bump `FOLLOWUP_VERSION`
  when a check changes, and the next run redoes everyone;
- **reuse**: when a candidate *is* redone, a transit fit or FPP stored for the same ephemeris is
  kept (`fu._reusable`); only the repetition searches and the difference image, where the rules
  evolve, are always recomputed. An FPP computed before background scenarios were included
  (no `fpp_background` field) is not reused, since it is optimistic.

## 3. The verdict rules

Each candidate gets one category from `fu.verdict(summary)`: the **first** check, in a fixed
order, that explains the signal. The order follows the strength of each piece of evidence:

1. **pixels off target → `blend`** (or `eb` if it is also deeper at 2P). A flux loss that beats
   what noise produces in the cutout and sits on a neighbour is decisive on its own: whatever the
   source does, it is not a planet on our star.
2. **no other sector could have shown it → `untestable`**. Repetition is the strongest test; without
   it there is no verdict, only a flag for more data.
3. **deeper at 2P and not repeating at P → `eb`**: two different eclipses folded together, a
   binary found at half its period.
4. **does not repeat → `not_repeating`**: informative sectors (expected SNR ≥ 7.5) did not return
   the period, or all other sectors combined did not.
5. **fitted stellar density outside 1/3–3× the catalogue value → `blend`** (or a wrong host star):
   the transit shape says the eclipsed object orbits a different star.
6. **FPP computed without the background scenarios → `incomplete`**: without TRILEGAL's simulated
   field TRICERATOPS drops the background-star scenarios and the FPP is optimistic (notebook 9 §3.4).
7. **FPP < 0.015 and NFPP < 0.001 → `validated`** (the TRICERATOPS validation level), **FPP < 0.5 →
   `candidate`** (worth a CTOI), **otherwise `ambiguous`**.
8. **fit or FPP missing → `incomplete`**.

When the FPP was computed with neighbours excluded by a per-pixel PRF fit (notebook 9 §3.6), that
FPP decides, and the one without exclusions is carried along (`FPP_without_exclusions`) and printed
in the report.

The pixel answer uses the `status` from `difference_image_stats` (notebook 9 §2.1): `on_target`,
`off_target` or `inconclusive`, with thresholds calibrated on pseudo-transit difference images.
*Inconclusive* pixels never reject a signal; they just leave the decision to the later rules.

The categories `not_repeating`, `eb` and `blend` are **rejected** (`fu.REJECTED`). The report keeps
these pages but ranks them last within their class, with the verdict printed on the page.

The rules can be checked on hand-made summaries, which is what `tests/test_followup_pipeline.py`
does. One example per category:

In [3]:
def toy(repeat=True, off=False, rho=None, fpp=None, other_sectors=True):
    """A minimal summary: a 2-d signal, two informative other sectors that do (or do not) return it."""
    P, depth = 2.0, 3000.0
    rows = [("s104", P, 9.0, depth, 9.0), ("all sectors", P, 12.0, depth, np.nan)]
    if other_sectors:
        p2 = P * 1.001 if repeat else 1.37
        rows[1:1] = [("s101", p2, 8.0, 2900, 9.0), ("s102", p2, 7.0, 3100, 9.0)]
    s = {"P": P, "T0": 0.0, "depth_ppm": depth, "discovery_sector": 104,
         "sector_search": [dict(zip(["data", "period", "snr", "depth_ppm", "snr_expected"], r)) for r in rows],
         "difference_image": {"offset_px": 2.0 if off else 0.1, "offset_err_px": 0.1,
                              "status": "off_target" if off else "on_target", "off_target": off}}
    if rho is not None:
        s["fit"] = {"rho_ratio_median": rho, "rp_earth": {"median": 2.0}}
    if fpp is not None:
        s["fpp"] = {"FPP": fpp[0], "NFPP": fpp[1], "fpp_background": True}
    return s


cases = {
    "repeats, on target, rho ok, FPP 0.01": toy(rho=1.0, fpp=(0.01, 0.0)),
    "repeats, on target, rho ok, FPP 0.2": toy(rho=1.0, fpp=(0.2, 0.01)),
    "repeats, on target, rho ok, FPP 0.8": toy(rho=1.0, fpp=(0.8, 0.8)),
    "repeats, on target, rho ok, FPP 0.01 without background": {**(x := toy(rho=1.0, fpp=(0.01, 0.0))),
        "fpp": {**x["fpp"], "fpp_background": False}},
    "repeats, on target, rho 0.1": toy(rho=0.1, fpp=(0.4, 0.4)),
    "repeats, OFF target": toy(off=True),
    "does not repeat": toy(repeat=False),
    "no other sector": toy(other_sectors=False),
    "repeats, on target, no fit/FPP yet": toy(),
}
pd.DataFrame({k: {c: fu.verdict(s)[c] for c in ["category", "verdict"]} for k, s in cases.items()}).T

,category,verdict
"repeats, on target, rho ok, FPP 0.01",validated,validated-level FPP
"repeats, on target, rho ok, FPP 0.2",candidate,planet candidate (CTOI-worthy)
"repeats, on target, rho ok, FPP 0.8",ambiguous,ambiguous (FPP >= 0.5)
"repeats, on target, rho ok, FPP 0.01 without background",incomplete,FPP without background scenarios (optimistic)
"repeats, on target, rho 0.1",blend,inconsistent stellar density: blend or wrong host
"repeats, OFF target",blend,flux loss off target: blend on a neighbour
does not repeat,not_repeating,not confirmed: does not repeat
no other sector,untestable,no verdict: no other sector could show it
"repeats, on target, no fit/FPP yet",incomplete,incomplete: fit or FPP missing


## 4. Running it

The stage runs as part of the full pipeline, after `known` and before `report`:

```bash
uv run --extra followup python -m src.cli run --sector 104                 # everything, follow-up included
uv run --extra followup python -m src.cli run --sector 104 --skip-followup # stop at the vetted list
uv run --extra followup python -m src.cli followup --sector 104            # just this stage
uv run python -m src.followup --sector 104 --verdict-only                  # re-apply the rules, no searches
uv run python -m src.followup --sector 104 --force                         # redo every candidate
uv run python -m src.followup --sector 104 --full-chain                    # fit + FPP even for rejected ones
uv run python -m src.followup --sector 104 --tics 214311908 --skip-fpp     # hand-run checks on chosen stars
```

How it behaves unattended:

- **parallel**: candidates run in parallel with joblib, at most half the cores by default (each
  worker runs TLS);
- **robust**: MAST, TESScut and Vizier time out now and then. Network errors are retried three
  times with growing waits; any other error is written into that candidate's `summary.json` and
  the table (`category = error`), and the run goes on. The next run retries only the failures;
- **outputs**: `data/followup/{tic}/summary.json` (everything measured, plus the verdict),
  `sectors.csv`, `diffimage_s{N}.npz`, `gaia.csv`, `fit.json`, and the sector table
  `data/followup/sector_{N}_followup.parquet`, which the report reads;
- **`--verdict-only`** recomputes every verdict from the stored summaries in seconds: for when the
  rules change, or another process has updated a fit or an FPP.

## 5. Sector 104

The stage has been run with `python -m src.cli followup --sector 104`. The cell below re-applies
the verdict rules to the stored summaries (`fu.refresh_verdicts`, the `--verdict-only` path), so
the notebook runs in seconds and always shows the current rules.

In [4]:
tab = fu.refresh_verdicts(SECTOR)
print("verdicts:", tab.category.value_counts().to_dict())
cols = ["tic_id", "category", "verdict", "repeats", "n_same_period", "n_other_sectors", "n_informative",
        "repetition_basis", "n_deeper_at_2P", "pixels", "density_ratio", "FPP", "FPP_std", "fpp_background",
        "n_fpp_excluded", "FPP_without_exclusions", "full_chain", "error"]
tab[cols].round(3)

verdicts: {'blend': 6, 'not_repeating': 5, 'candidate': 1, 'eb': 1}


,tic_id,category,verdict,repeats,n_same_period,n_other_sectors,n_informative,repetition_basis,n_deeper_at_2P,pixels,density_ratio,FPP,FPP_std,fpp_background,n_fpp_excluded,FPP_without_exclusions,full_chain,error
0,214311908,candidate,planet candidate (CTOI-worthy),True,3,5,2,sectors,0,on,0.940,0.089,0.053,True,4,0.45,True,None
1,161313862,not_repeating,not confirmed: does not repeat,False,0,6,4,sectors,0,on,0.198,0.300,0.024,True,0,NaN,False,None
2,231888176,not_repeating,not confirmed: does not repeat,False,0,5,5,sectors,0,inconclusive,NaN,NaN,NaN,None,0,NaN,False,None
3,372868160,not_repeating,not confirmed: does not repeat,False,0,5,4,sectors,0,inconclusive,NaN,NaN,NaN,None,0,NaN,False,None
4,389127976,not_repeating,not confirmed: does not repeat,False,0,9,9,sectors,0,inconclusive,NaN,NaN,NaN,None,0,NaN,False,None
5,394585043,not_repeating,not confirmed: does not repeat,False,0,9,0,combined,0,inconclusive,0.062,0.074,0.009,True,0,NaN,False,None
6,424913827,eb,"eclipsing binary on a neighbour, found at half its period",False,2,4,0,sectors,1,off,NaN,NaN,NaN,None,0,NaN,False,None
7,61615837,blend,flux loss off target: blend on a neighbour,False,1,3,3,sectors,0,off,NaN,NaN,NaN,None,0,NaN,False,None
8,64336445,blend,inconsistent stellar density: blend or wrong host,True,0,4,0,combined,0,inconclusive,0.101,0.494,0.016,True,0,NaN,True,None
9,88516648,blend,flux loss off target: blend on a neighbour,False,1,3,3,sectors,0,off,NaN,NaN,NaN,None,0,NaN,False,None


### 5.1 Compared with the verdicts reached by hand

`docs/REVIEW_sector104.md` graded every signal by eye and then followed up the survivors with
notebook 9. Its outcome for the 13 new signals is the reference:

In [5]:
review = {
    306101698: ("blend", "EB on a neighbour (repeats, loss 2 px away)"),
    229804549: ("blend", "EB on a neighbour (loss 6 px away)"),
    424913827: ("eb", "EB on a neighbour, found at half its period"),
    61615837: ("blend", "EB on a neighbour (loss 2.7 px away)"),
    88516648: ("blend", "blend (loss 2.4 px away)"),
    64336445: ("blend", "likely blend: stellar density 0.10x"),
    218663210: ("blend", "V-shaped, centroid 59 sigma: blended EB (by eye)"),
    372868160: ("not_repeating", "does not repeat"),
    161313862: ("not_repeating", "does not repeat"),
    394585043: ("not_repeating", "does not repeat (all other sectors combined)"),
    231888176: ("systematic", "glitch-like drops in orbit 2 (by eye)"),
    389127976: ("systematic", "20 % flux steps in the raw data (by eye)"),
    214311908: ("ambiguous", "unresolved: repeats, on target, 3-5\" neighbours"),
}
cmp = tab.set_index("tic_id")[["category", "verdict"]].rename(columns={"category": "automatic", "verdict": "automatic verdict"})
cmp.insert(0, "by hand", pd.Series({t: r[0] for t, r in review.items()}))
cmp["by hand, reason"] = pd.Series({t: r[1] for t, r in review.items()})
rejected = fu.REJECTED | {"systematic"}
cmp["agree"] = [(a == h) or (a in fu.REJECTED and h in rejected) for a, h in zip(cmp.automatic, cmp["by hand"])]
cmp

,by hand,automatic,automatic verdict,"by hand, reason",agree
tic_id,,,,,
214311908,ambiguous,candidate,planet candidate (CTOI-worthy),"unresolved: repeats, on target, 3-5"" neighbours",False
161313862,not_repeating,not_repeating,not confirmed: does not repeat,does not repeat,True
231888176,systematic,not_repeating,not confirmed: does not repeat,glitch-like drops in orbit 2 (by eye),True
372868160,not_repeating,not_repeating,not confirmed: does not repeat,does not repeat,True
389127976,systematic,not_repeating,not confirmed: does not repeat,20 % flux steps in the raw data (by eye),True
394585043,not_repeating,not_repeating,not confirmed: does not repeat,does not repeat (all other sectors combined),True
424913827,eb,eb,"eclipsing binary on a neighbour, found at half its period","EB on a neighbour, found at half its period",True
61615837,blend,blend,flux loss off target: blend on a neighbour,EB on a neighbour (loss 2.7 px away),True
64336445,blend,blend,inconsistent stellar density: blend or wrong host,likely blend: stellar density 0.10x,True


**Every signal the review rejected is rejected automatically (12 of 12)**, and the one it left
open comes out on top, now as a candidate rather than "ambiguous" (the one `False` above):

- **the seven blends and eclipsing binaries** are all caught. Five by the difference image alone
  (the flux loss sits 2–6 px away, on a neighbour), TIC 424913827 additionally as a binary found
  at half its period (deeper at 2P), and TIC 64336445, whose pixels are inconclusive, by its
  transit shape: the fitted stellar density is 0.10× the target's. TIC 218663210 was rejected by
  eye for its V shape; the stage finds the reason, a loss 2 px away at S/N 20;
- **the three non-repeating signals** do not repeat. TIC 394585043 is the instructive one: no
  single other sector could have shown it, so the decision comes from all other sectors combined
  (`repetition_basis = combined`);
- **the two systematics** the review spotted by eye (glitch-like drops, flux steps) come out as
  *not repeating*, with 5 and 9 informative sectors. The stage has no notion of "systematic",
  but a systematic does not recur at the same period, so repetition catches it;
- **TIC 214311908** is the only signal that repeats, sits on the target (barely: S/N 4.3 against a
  noise threshold of 4.25 with 4 transits) and has the right stellar density. Its FPP, 0.09 ± 0.05
  with the neighbours a per-pixel fit rules out removed (0.45 ± 0.14 without), puts it below the
  0.5 bar: the rules call it a **candidate**. The review called it ambiguous because its FPP was
  0.64 at the time; the change comes from the FPP, not from the rules (notebook 9 §3.4–3.6, and
  `notebooks/candidate-TIC214311908.ipynb` for the full case).

The cheap rejection did its job: the fit and FPP ran only where repetition and pixels left the
question open (`full_chain`).

### 5.2 In the report

`python -m src.cli report` joins the table to the vetted signals. Each candidate page gets a
coloured follow-up line under the header (green: candidate or validated, orange: ambiguous,
incomplete or untestable, red: rejected) with the numbers behind it, and the summary page counts
the verdicts. Within each class, pages are ordered by verdict, so rejected signals sink to the end
of the `new` block instead of sitting at the top because of a high SNR.

In [6]:
rep = json.loads((DATA_DIR / "reports" / f"sector_{SECTOR}_summary.json").read_text())
print("follow-up verdicts in the report summary:", rep.get("followup"))
pd.read_csv(DATA_DIR / "reports" / f"sector_{SECTOR}_candidates.csv").query(
    "known_class in ['new', 'new_in_known_system']")[["rank", "tic_id", "period", "rp_earth", "snr", "fu_category", "fu_verdict"]]

follow-up verdicts in the report summary: {'blend': 6, 'not_repeating': 5, 'candidate': 1, 'eb': 1}


,rank,tic_id,period,rp_earth,snr,fu_category,fu_verdict
0,1,214311908,5.692444,4.385473,10.161878,candidate,planet candidate (CTOI-worthy)
1,2,394585043,4.495943,2.422301,6.556242,not_repeating,not confirmed: does not repeat
2,3,161313862,0.292860,1.230182,7.905993,not_repeating,not confirmed: does not repeat
3,4,231888176,5.314070,1.650576,6.347133,not_repeating,not confirmed: does not repeat
4,5,372868160,4.512895,1.483066,7.565244,not_repeating,not confirmed: does not repeat
5,6,389127976,0.576410,5.930242,7.131763,not_repeating,not confirmed: does not repeat
6,7,424913827,1.572835,0.922658,7.052057,eb,"eclipsing binary on a neighbour, found at half its period"
7,8,229804549,0.732053,2.755067,17.299870,blend,flux loss off target: blend on a neighbour
8,9,306101698,2.878971,3.065793,10.459270,blend,flux loss off target: blend on a neighbour
9,10,61615837,1.107593,2.358681,10.260555,blend,flux loss off target: blend on a neighbour


## 6. Limits

- **Automatic verdicts are not final.** The rules encode what notebook 9 decided by hand, and they
  agree with it on sector 104, but every new sector can bring a case the rules did not foresee. The
  report shows the numbers behind each verdict so a person can overrule it.
- **Systematics are not a category.** The stage has no test for "glitch-like drops" or "flux steps":
  such signals come out as whatever repetition says (usually not repeating, since a systematic does
  not recur at the same period). The by-eye grades of the review remain useful.
- **Pixels can only reject.** An `on_target` difference image means the loss is within ~0.3 px
  (6″) of the star. A neighbour closer than that cannot be excluded with TESS; that is what the
  FPP, and in the end ground-based photometry, are for (TIC 214311908, notebook
  `candidate-TIC214311908`).
- **The FPP depends on the background model.** TRICERATOPS' background scenarios use a TRILEGAL
  simulation of the field; FPPs computed without it are lower than they should be, and the
  summaries record which way each was computed (`fpp_background`, `fpp_method`). TRICERATOPS is
  also random: the stage reports the mean and scatter of 5 runs, and the scatter (up to ±0.14)
  is often larger than the differences between candidates.